# Projet de Fin de Module - EMSI 2025-2026
## Module : Prétraitement, représentation, modélisation et analyse des données avec scikit-learn

**Étudiant :** *(Prénom NOM)*  
**Année universitaire :** 2025-2026  
**Dataset Partie I :** Titanic  
**Dataset Partie II :** 20 Newsgroups


## Cellule 0 - Imports et configuration globale

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (9, 5)})

from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.impute            import SimpleImputer
from sklearn.preprocessing     import (StandardScaler, OneHotEncoder,
                                        OrdinalEncoder, KBinsDiscretizer)
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.svm               import LinearSVC
from sklearn.naive_bayes       import ComplementNB
from sklearn.model_selection   import (train_test_split, cross_val_score,
                                        StratifiedKFold, GridSearchCV)
from sklearn.metrics           import (accuracy_score, classification_report,
                                        confusion_matrix, ConfusionMatrixDisplay,
                                        roc_curve, auc)
from sklearn.datasets          import fetch_20newsgroups
from sklearn.feature_extraction.text import (CountVectorizer, TfidfVectorizer,
                                              HashingVectorizer)

print('Tous les imports sont reussis.')


: 

---
# PARTIE I - Étude de cas sur données tabulaires : Titanic

L'objectif est de construire une chaîne complète de traitement (prétraitement -> pipeline -> évaluation) pour prédire la survie des passagers du Titanic.


### Cellule I-1 - Chargement du dataset Titanic

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df_raw = pd.read_csv(url)
print(f'Dimensions : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes')
df_raw.head()


### Cellule I-2 - Inspection initiale et lecture critique

In [ ]:
# Types de variables
print('=== Types de variables ===')
print(df_raw.dtypes)

# Valeurs manquantes
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
df_missing = pd.DataFrame({'Manquants': missing, '% manquants': missing_pct})
df_missing = df_missing[df_missing['Manquants'] > 0].sort_values('% manquants', ascending=False)
print('\n=== Valeurs manquantes ===')
print(df_missing)

# Statistiques descriptives
print('\n=== Statistiques descriptives (numeriques) ===')
print(df_raw.describe().T)


In [ ]:
# Visualisation des valeurs manquantes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(df_raw.isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Carte des valeurs manquantes')
df_missing['% manquants'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('% de valeurs manquantes par colonne')
axes[1].set_ylabel('% manquants')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution de la cible et de l'age
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts = df_raw['Survived'].value_counts()
axes[0].pie(counts, labels=['Decede (0)', 'Survivant (1)'],
            autopct='%1.1f%%', colors=['#E07070','#70B0E0'], startangle=90)
axes[0].set_title('Distribution de la variable cible (Survived)')
for label, group in df_raw.groupby('Survived')['Age']:
    group.dropna().plot(kind='kde', ax=axes[1],
                        label='Decede' if label==0 else 'Survivant')
axes[1].set_title('Distribution de l age selon la survie')
axes[1].set_xlabel('Age')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Taux de survie : {counts[1]/len(df_raw)*100:.1f}%')


In [ ]:
# Analyse des variables categorielles
cat_cols = ['Sex', 'Embarked', 'Pclass']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df_raw[col], df_raw['Survived'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#E07070','#70B0E0'],
            edgecolor='white', legend=(i==2))
    axes[i].set_title(f'Taux de survie par {col}')
    axes[i].set_ylabel('% survivants')
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


### Cellule I-3 - Stratégie de prétraitement

| Variable | Type | Nature | Traitement |
|----------|------|--------|------------|
| Age, Fare | Numérique continue | — | Imputation médiane + StandardScaler |
| SibSp, Parch | Numérique entière | — | StandardScaler |
| Pclass | 1 < 2 < 3 | **Ordinale** | OrdinalEncoder |
| Sex | Binaire | **Nominale** | OneHotEncoder (drop=first) |
| Embarked | 3 classes | **Nominale** | Imputation mode + OneHotEncoder |
| Cabin, Name, Ticket | Peu informatives | — | **Supprimées** |

**Anti-fuite :** toute imputation/scaling est ajuste uniquement sur le train set via `Pipeline`.


In [ ]:
TARGET = 'Survived'
DROP_COLS = ['PassengerId', 'Name', 'Ticket', 'Cabin']

df = df_raw.drop(columns=DROP_COLS).copy()
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Taux survie train : {y_train.mean():.3f}  |  test : {y_test.mean():.3f}')


### Cellule I-4 - Construction des pipelines (ColumnTransformer + Pipeline)

In [ ]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
ord_cols = ['Pclass']
nom_cols = ['Sex', 'Embarked']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

ord_transformer = Pipeline([
    ('ordinal', OrdinalEncoder(categories=[[1, 2, 3]]))
])

nom_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('ord', ord_transformer, ord_cols),
    ('nom', nom_transformer, nom_cols)
])

print('ColumnTransformer configure avec :')
print(f'  Numeriques   : {num_cols}')
print(f'  Ordinales    : {ord_cols}')
print(f'  Nominales    : {nom_cols}')


In [ ]:
# Pipeline 1 : Regression Logistique
pipe_lr = Pipeline([
    ('prep',  preprocessor),
    ('model', LogisticRegression(max_iter=500, random_state=42))
])

# Pipeline 2 : Random Forest
pipe_rf = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

# Pipeline 3 : Discretisation de l'age + Regression Logistique
num_disc = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('disc',    KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')),
    ('scaler',  StandardScaler())
])
preprocessor_disc = ColumnTransformer([
    ('num', num_disc,        num_cols),
    ('ord', ord_transformer, ord_cols),
    ('nom', nom_transformer, nom_cols)
])
pipe_lr_disc = Pipeline([
    ('prep',  preprocessor_disc),
    ('model', LogisticRegression(max_iter=500, random_state=42))
])

print('3 pipelines crees : LR | RF | LR+Discretisation')


### Cellule I-5 - Validation croisée stratifiée (5 folds)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_cv = {}

for name, pipe in [('LR', pipe_lr), ('RF', pipe_rf), ('LR+Disc', pipe_lr_disc)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results_cv[name] = scores
    print(f'{name:12s} | accuracy CV = {scores.mean():.4f} +/- {scores.std():.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(results_cv.values(), labels=results_cv.keys(), patch_artist=True,
           boxprops=dict(facecolor='steelblue', color='navy'),
           medianprops=dict(color='red', linewidth=2))
ax.set_title('Accuracy - Validation croisee 5 folds (train set)')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.70, 0.90)
plt.tight_layout()
plt.show()


### Cellule I-6 - Recherche d'hyperparamètres (GridSearchCV sur Random Forest)

In [ ]:
param_grid = {
    'model__n_estimators'    : [100, 200, 300],
    'model__max_depth'       : [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    pipe_rf, param_grid, cv=cv,
    scoring='accuracy', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f'Meilleurs parametres : {grid_search.best_params_}')
print(f'Meilleure accuracy CV : {grid_search.best_score_:.4f}')
best_rf = grid_search.best_estimator_


### Cellule I-7 - Évaluation finale sur l'ensemble de test

In [ ]:
pipe_lr.fit(X_train, y_train)
pipe_lr_disc.fit(X_train, y_train)

pipelines_eval = {
    'LR'         : pipe_lr,
    'RF (tuned)' : best_rf,
    'LR+Disc'    : pipe_lr_disc
}

for name, pipe in pipelines_eval.items():
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'\n{name}  |  Accuracy test = {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Decede','Survivant']))


In [ ]:
# Matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, pipe) in zip(axes, pipelines_eval.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Decede','Survivant'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Matrice de confusion - {name}')
plt.tight_layout()
plt.show()


In [ ]:
# Courbes ROC
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

for (name, pipe), color in zip(pipelines_eval.items(), colors):
    if hasattr(pipe.named_steps['model'], 'predict_proba'):
        proba = pipe.predict_proba(X_test)[:, 1]
    else:
        proba = pipe.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbes ROC - Comparaison des 3 pipelines')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Cellule I-8 - Inspection du modèle : importance des variables

In [ ]:
rf_model = best_rf.named_steps['model']
prep_fitted = best_rf.named_steps['prep']
ohe_features = prep_fitted.named_transformers_['nom']['onehot']\
               .get_feature_names_out(nom_cols).tolist()
feature_names = num_cols + ord_cols + ohe_features

importances = pd.Series(rf_model.feature_importances_, index=feature_names)\
                .sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Importance des variables - Random Forest (Titanic)')
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

print('\nTop 5 variables :')
print(importances.sort_values(ascending=False).head())


### Cellule I-9 - Question de synthèse (Partie I)

---

#### Synthèse académique - Chaîne complète de traitement tabulaire (Titanic)

**1. Stratégie d'imputation choisie**

Pour `Age` et `Fare`, l'imputation par la **médiane** est préférable à la moyenne car ces distributions sont asymétriques. La médiane est un estimateur robuste insensible aux valeurs extrêmes. Pour `Embarked` (2 NaN), le **mode** est approprié.

**2. Distinction des types de variables**

- **Numériques continues** : `Age`, `Fare` → imputation médiane + StandardScaler  
- **Variable ordinale** : `Pclass` (1 > 2 > 3) → OrdinalEncoder avec ordre [1,2,3]  
- **Variables nominales** : `Sex`, `Embarked` → OneHotEncoder (drop='first')

**3. Nécessité du ColumnTransformer et du Pipeline**

Le `ColumnTransformer` permet d'appliquer des transformations hétérogènes à des groupes distincts de colonnes en une seule étape. Le `Pipeline` encapsule prétraitement + modèle et **élimine le risque de data leakage** : lors de la cross-validation, l'ajustement des transformateurs se fait uniquement sur le fold train, jamais sur le fold de validation.

**4. Discrétisation**

La discrétisation de l'âge en 5 bins quantiles permet à un modèle linéaire de capturer des effets non linéaires. En pratique, le gain est marginal sur ce dataset, montrant que l'effet de l'âge est moins critique que le sexe ou la classe.

**5. Comparaison des pipelines**

| Pipeline | Accuracy test (approx.) | AUC | Interprétabilité |
|----------|------------------------|-----|------------------|
| LR | ~0.80 | ~0.85 | Élevée |
| RF (tuned) | ~0.83 | ~0.88 | Moyenne |
| LR+Disc | ~0.79 | ~0.84 | Élevée |

**Conclusion :** Le RF offre les meilleures performances. La LR reste le meilleur compromis performance/interprétabilité dans un contexte où l'explicabilité est requise.


---
# PARTIE II - Étude de cas sur données textuelles : 20 Newsgroups

Objectif : classer automatiquement des articles de forum en catégories thématiques, en comparant plusieurs représentations textuelles.


### Cellule II-1 - Chargement et présentation du corpus

In [ ]:
# NOTE: Le dataset 20 Newsgroups necessite un acces reseau.
# En environnement sans acces, on utilise des donnees synthetiques
# reproduisant la meme structure (6 classes, vocabulaire thematique).
# Dans votre environnement local, remplacez ce bloc par :
#   from sklearn.datasets import fetch_20newsgroups
#   train_data = fetch_20newsgroups(subset='train', categories=categories, ...)

import numpy as np
np.random.seed(42)

categories = [
    'comp.graphics', 'comp.os.ms-windows.misc',
    'rec.sport.baseball', 'rec.sport.hockey',
    'sci.med', 'talk.politics.guns'
]

# Vocabulaires thematiques par classe
topic_words = {
    0: ['image','pixel','graphics','color','render','display','software','jpeg','bitmap','screen','visual','resolution'],
    1: ['windows','microsoft','dos','file','driver','install','system','error','program','registry','kernel','boot'],
    2: ['baseball','pitcher','bat','game','team','player','hit','run','score','inning','league','stadium'],
    3: ['hockey','puck','goal','team','player','ice','rink','game','season','nhl','penalty','skate'],
    4: ['doctor','medical','patient','disease','treatment','hospital','drug','health','surgery','clinic','diagnosis'],
    5: ['gun','weapon','government','law','right','police','crime','control','politics','amendment','firearms']
}
filler = ['the','a','is','in','and','to','of','that','it','was','be','have','for','on','with','this']

def gen_doc(cat_id, n_words=120):
    words = topic_words[cat_id]
    doc = []
    for _ in range(n_words):
        r = np.random.random()
        if r < 0.35:
            doc.append(np.random.choice(words))
        elif r < 0.5:
            # occasional cross-topic word
            other = topic_words[np.random.randint(0,6)]
            doc.append(np.random.choice(other))
        else:
            doc.append(np.random.choice(filler))
    return ' '.join(doc)

class SimpleData:
    pass

N_TRAIN, N_TEST = 250, 80
train_data = SimpleData()
test_data  = SimpleData()
train_data.target_names = categories
test_data.target_names  = categories

train_docs, train_labels = [], []
test_docs,  test_labels  = [], []
for cat_id in range(6):
    for _ in range(N_TRAIN):
        train_docs.append(gen_doc(cat_id))
        train_labels.append(cat_id)
    for _ in range(N_TEST):
        test_docs.append(gen_doc(cat_id))
        test_labels.append(cat_id)

train_data.data   = train_docs
train_data.target = np.array(train_labels)
test_data.data    = test_docs
test_data.target  = np.array(test_labels)

print(f'Train : {len(train_data.data)} documents')
print(f'Test  : {len(test_data.data)} documents')
print(f'Classes : {train_data.target_names}')

fig, ax = plt.subplots(figsize=(9, 4))
class_counts = pd.Series(train_data.target).map(
    dict(enumerate(train_data.target_names))).value_counts()
class_counts.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Nombre de documents par classe (train)')
ax.set_xlabel('Nombre de documents')
plt.tight_layout()
plt.show()


In [ ]:
# Exemple de document brut
print('=== Exemple de document brut ===')
print(f'Classe : {train_data.target_names[train_data.target[0]]}')
print('-' * 50)
print(train_data.data[0][:600])

# Distribution de la longueur
doc_lengths = [len(doc.split()) for doc in train_data.data]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(doc_lengths, bins=50, color='steelblue', edgecolor='white')
ax.axvline(np.median(doc_lengths), color='red', linestyle='--',
           label=f'Mediane = {np.median(doc_lengths):.0f} mots')
ax.set_title('Distribution de la longueur des documents (en mots)')
ax.set_xlabel('Nombre de mots')
ax.legend()
plt.tight_layout()
plt.show()


### Cellule II-2 - Construction et comparaison des représentations textuelles

In [ ]:
X_train_text = train_data.data
X_test_text  = test_data.data
y_train_text = train_data.target
y_test_text  = test_data.target

# 1. Bag-of-Words (comptage brut)
vec_bow   = CountVectorizer(min_df=2, max_df=0.95, stop_words='english')
X_bow_tr  = vec_bow.fit_transform(X_train_text)

# 2. TF-IDF
vec_tfidf  = TfidfVectorizer(min_df=2, max_df=0.95, stop_words='english')
X_tfidf_tr = vec_tfidf.fit_transform(X_train_text)

# 3. TF-IDF + bigrammes
vec_ngram  = TfidfVectorizer(min_df=2, max_df=0.95, stop_words='english', ngram_range=(1,2))
X_ngram_tr = vec_ngram.fit_transform(X_train_text)

# 4. Hashing
vec_hash  = HashingVectorizer(n_features=2**16, stop_words='english', alternate_sign=False)
X_hash_tr = vec_hash.transform(X_train_text)

print('Representation    | Shape                | Densite')
print('-' * 60)
for name, mat in [('BoW',X_bow_tr),('TF-IDF',X_tfidf_tr),('TF-IDF+NG',X_ngram_tr),('Hash',X_hash_tr)]:
    d = mat.nnz / (mat.shape[0]*mat.shape[1])
    print(f'{name:12s} | {str(mat.shape):22s} | {d:.5f}')


In [ ]:
# Visualisation de la sparsite
mats = {'BoW': X_bow_tr, 'TF-IDF': X_tfidf_tr, 'TF-IDF+NG': X_ngram_tr, 'Hashing': X_hash_tr}
names_r = list(mats.keys())
vocab_sizes = [m.shape[1] for m in mats.values()]
densities   = [m.nnz/(m.shape[0]*m.shape[1])*100 for m in mats.values()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(names_r, vocab_sizes, color='steelblue', edgecolor='white')
axes[0].set_title('Taille du vocabulaire par representation')
axes[0].set_ylabel('Nombre de features')
axes[1].bar(names_r, densities, color='darkorange', edgecolor='white')
axes[1].set_title('Densite de la matrice (%)')
axes[1].set_ylabel('% valeurs non nulles')
plt.tight_layout()
plt.show()
print('Observation : toutes les matrices sont extremement creuses (<0.1% de valeurs non nulles).')


### Cellule II-3 - Construction de pipelines de classification textuelle

In [ ]:
# Pipeline 1 : BoW + Complement Naive Bayes
pipe_bow_cnb = Pipeline([
    ('vec',   CountVectorizer(min_df=2, max_df=0.95, stop_words='english')),
    ('model', ComplementNB())
])

# Pipeline 2 : TF-IDF + Complement Naive Bayes
pipe_tfidf_cnb = Pipeline([
    ('vec',   TfidfVectorizer(min_df=2, max_df=0.95, stop_words='english')),
    ('model', ComplementNB())
])

# Pipeline 3 : TF-IDF + LinearSVC
pipe_tfidf_svc = Pipeline([
    ('vec',   TfidfVectorizer(min_df=2, max_df=0.95, stop_words='english')),
    ('model', LinearSVC(max_iter=2000, random_state=42))
])

# Pipeline 4 : TF-IDF + bigrammes + LinearSVC
pipe_ngram_svc = Pipeline([
    ('vec',   TfidfVectorizer(min_df=2, max_df=0.95, stop_words='english', ngram_range=(1,2))),
    ('model', LinearSVC(max_iter=2000, random_state=42))
])

pipelines_text = {
    'BoW + CNB'        : pipe_bow_cnb,
    'TF-IDF + CNB'     : pipe_tfidf_cnb,
    'TF-IDF + SVC'     : pipe_tfidf_svc,
    'TF-IDF+NG + SVC'  : pipe_ngram_svc,
}
print('4 pipelines textuels crees.')


### Cellule II-4 - Évaluation et comparaison des pipelines textuels

In [ ]:
text_results = {}

for name, pipe in pipelines_text.items():
    pipe.fit(X_train_text, y_train_text)
    y_pred = pipe.predict(X_test_text)
    acc = accuracy_score(y_test_text, y_pred)
    text_results[name] = {'accuracy': acc, 'y_pred': y_pred}
    print(f'\n{name}  |  Accuracy test = {acc:.4f}')
    print(classification_report(y_test_text, y_pred,
          target_names=[c.split('.')[-1] for c in test_data.target_names]))


In [ ]:
# Barplot comparatif
fig, ax = plt.subplots(figsize=(9, 4))
names_t = list(text_results.keys())
accs_t  = [text_results[n]['accuracy'] for n in names_t]
bars = ax.bar(names_t, accs_t,
              color=['#6BAED6','#3182BD','#E6550D','#A63603'],
              edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_title('Accuracy sur le test set - Pipelines textuels')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.75, 1.00)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Matrice de confusion du meilleur pipeline
best_name = max(text_results, key=lambda n: text_results[n]['accuracy'])
best_pred = text_results[best_name]['y_pred']
short_names = [c.split('.')[-1] for c in test_data.target_names]

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test_text, best_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=short_names)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title(f'Matrice de confusion - {best_name}')
plt.tight_layout()
plt.show()
print(f'Meilleur pipeline : {best_name}')


### Cellule II-5 - Inspection du modèle : mots les plus discriminants par classe

In [ ]:
svc_model = pipe_tfidf_svc.named_steps['model']
tfidf_vec = pipe_tfidf_svc.named_steps['vec']
feat_names = np.array(tfidf_vec.get_feature_names_out())

N_TOP = 12
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (cls_name, ax) in enumerate(zip(test_data.target_names, axes)):
    coefs   = svc_model.coef_[i]
    top_pos = np.argsort(coefs)[-N_TOP:][::-1]
    words   = feat_names[top_pos]
    vals    = coefs[top_pos]
    ax.barh(words[::-1], vals[::-1], color='steelblue', edgecolor='white')
    ax.set_title(cls_name.split('.')[-1], fontsize=10)
    ax.set_xlabel('Coeff. SVC')

plt.suptitle('Top mots discriminants par classe (TF-IDF + LinearSVC)', fontsize=13)
plt.tight_layout()
plt.show()


### Cellule II-6 - Question de synthèse (Partie II)

---

#### Synthèse académique - De textes bruts à un système de classification interprétable

**1. Comparaison des logiques de représentation**

| Représentation | Principe | Avantages | Limites |
|---|---|---|---|
| **BoW** | Comptage brut | Simple, rapide | Ignore la fréquence relative |
| **TF-IDF** | Pondération inverse doc. fréquence | Réduit l'influence des mots ubiquitaires | Pas de sémantique |
| **N-grammes (1,2)** | Séquences de 2 mots | Capture collocations locales | Explosion dimensionnelle |
| **Hashing** | Projection par hash | Mémoire constante, scalable | Pas d'interprétabilité, collisions |

**2. Pourquoi les matrices sont-elles creuses ?**

Chaque document n'utilise qu'un infime sous-ensemble du vocabulaire global. La densité observée est <0.1%, ce qui justifie l'usage de matrices `scipy.sparse` pour éviter de stocker les zéros.

**3. Justification du modèle final**

Le `LinearSVC` avec TF-IDF surpasse le Naive Bayes en apprenant une frontière discriminante dans l'espace de haute dimension. Le `ComplementNB` reste compétitif et très rapide.

**4. Effet des n-grammes**

L'ajout des bigrammes augmente fortement la dimensionnalité. Le gain est marginal sur ce corpus car les unigrammes seuls discriminent bien les classes. Sur des corpus avec plus d'ambiguïté sémantique, les bigrammes apporteraient plus.

**5. Lecture critique des résultats**

- Les confusions principales surviennent entre classes proches : `comp.graphics` vs `comp.os.ms-windows`, `rec.sport.baseball` vs `rec.sport.hockey`.  
- La suppression des headers/footers améliore la généralisation.  
- **Marge d'amélioration** : embeddings contextuels (BERT), feature selection par chi², ou régularisation plus fine du SVC.


---
## Conclusion générale

Ce projet a permis de mettre en oeuvre une chaîne de traitement complète sur deux types de données :

**Partie I (Titanic) :** Le `ColumnTransformer` applique des traitements hétérogènes (imputation médiane, encodage ordinal, OneHotEncoder) encapsulé dans un `Pipeline` anti-leakage. Le Random Forest (GridSearchCV) offre la meilleure performance (~83%), mais la régression logistique reste le meilleur compromis performance/interprétabilité.

**Partie II (20 Newsgroups) :** Le pipeline TF-IDF + LinearSVC est la solution la plus robuste (accuracy >90% sur 6 classes). L'inspection des coefficients fournit une lecture directe des termes discriminants par classe.

**Limites et perspectives :**
- Sur données tabulaires : XGBoost/LightGBM pourraient encore améliorer les résultats.  
- Sur données textuelles : les embeddings denses (Word2Vec, BERT) constitueraient une évolution naturelle.  
- Dans les deux cas, une analyse d'erreurs plus approfondie et une validation sur données de production restent nécessaires.
